# General Data Science Track — Day 3: Dates and longitudinal data
**Notebook / task:** 3  
**Estimated core time:** 60 minutes  
**Curriculum:** Phase 1.3 — Dates, times, time zones, durations, rolling windows, and longitudinal shape

## Why this matters
**How did recorded purchase activity change during 2025, and how long were the gaps between a customer's purchases?** Date parsing alone cannot answer this: you must define the observation window, preserve customer boundaries, and distinguish elapsed time from calendar labels. Build a descriptive analysis from the local transaction data, then use a tiny timestamp example to understand daylight-saving time. This notebook runs independently of earlier lessons.

## Learning objectives
By the end, you will be able to:
- parse dates while exposing failed conversions and filter an explicit observation window;
- distinguish date-only values, UTC instants, and local clock times;
- calculate within-customer gaps and recency at a fixed reference date;
- build a complete daily calendar and a trailing seven-day activity summary; and
- explain what a descriptive time plot can and cannot establish.

## R-to-Python bridge
| R / lubridate / tidyverse | pandas |
|---|---|
| `ymd(x)` | `pd.to_datetime(x, format="%Y-%m-%d", errors="coerce")` |
| `year(x)`, `month(x)` | `series.dt.year`, `series.dt.month` |
| `with_tz(x, "Europe/Vienna")` | `series.dt.tz_convert("Europe/Vienna")` for aware timestamps |
| Attach a zone to naive clock values | `series.dt.tz_localize(...)`; ambiguous/nonexistent times need an explicit policy |
| `arrange(id, date)` + grouped `lag()` | `.sort_values(...)` + `.groupby(id)[date].shift()` or `.diff()` |
| `difftime()` | timestamp subtraction produces timedeltas; `.dt.total_seconds()` or `.dt.days` |
| Complete a calendar | `pd.date_range(...)` + `.reindex(...)` |
| Trailing window | `.rolling(window=7, min_periods=7).mean()` on a complete daily Series |

A Series uses `.dt` for datetime properties; a `DatetimeIndex` exposes them directly. `.dt.days` gives whole-day components; use `.dt.total_seconds() / 3600` for elapsed hours. Date-only purchase records have **no recorded time zone or time of day**: do not manufacture either.

## Suggested pacing
- Tasks 1–3: load, parse, and define the window — 13 minutes
- Tasks 4–6: time zones, gaps, and recency — 19 minutes
- Tasks 7–9: daily calendar, rolling summary, and plot — 18 minutes
- Retrieval practice — 5 minutes
- Reasoning and reflection — 5 minutes

### Inputs and analysis contract
Start Jupyter from the repository root. Use `data/transactions.csv`, generated locally by `agent/generate_learning_data.py` with the repository's fixed seed. No download or earlier notebook variables are required.

- Raw shape: **12,091 rows × 7 columns**; one row is one transaction, keyed by `transaction_id`. `customer_id` repeats.
- Dates are synthetic calendar dates. The generator's observation interval is **2024-01-01 inclusive through 2025-12-31 exclusive**; December 31, 2025 is outside coverage.
- Analyze **2025-01-01 inclusive through 2025-12-31 exclusive**, a 364-day window. Use **2025-12-31** as the fixed reference date for recency.
- Count **all recorded transactions**, including returned purchases; call the result transaction activity, not net sales. You do not need amounts or discounts today.
- For this exercise, assume capture is complete within the stated interval. Only under that assumption may an absent daily row become zero transactions.
- Keep `raw` unchanged. Use a copy named `tx` for parsing and a copy named `analysis` for the selected period. No joins are needed.

Write explanations in new markdown cells or print a short response. Exercise cells are intentionally empty. API reminders are scaffolding, not code you must copy verbatim.

## Task 1 — Load and predict (4 minutes)
Import pandas as `pd`, and load the CSV into `raw`. Inspect shape, dtypes, and key uniqueness. Before executing, predict the type and shape of `raw["transaction_date"]` versus `raw[["transaction_date"]]`, and predict the date column's dtype after ordinary CSV loading.

Confirm your predictions. Assert the raw shape and a nonmissing, unique transaction key; explain why `customer_id` is not a transaction key. Create `tx` as a copy of `raw`.

In [3]:
import pandas as pd

raw = pd.read_csv("data/transactions.csv")

raw.head()

,transaction_id,customer_id,transaction_date,category,amount_eur,discount_pct,returned
0,T000001,C00001,2025-03-23,hardware,118.37,0.0,False
1,T000002,C00001,2024-08-22,software,78.01,0.0,False
2,T000003,C00001,2025-11-23,training,103.17,0.0,False
3,T000004,C00001,2024-10-29,software,79.45,0.0,False
4,T000005,C00001,2025-06-30,software,249.17,0.0,False


In [15]:
print(raw.shape)
print(raw.dtypes)
print(raw["transaction_id"].nunique())

(12091, 7)
transaction_id          str
customer_id             str
transaction_date        str
category                str
amount_eur          float64
discount_pct        float64
returned               bool
dtype: object
12091


As predicted. Numeric columns are correct. String columns are mostly correct. transaciton date however should be transformte to a pd datetime opbject. customers can do multiple transaciton, so these must not be unique.

In [18]:
tx = raw.copy()

## Task 2 — Parse with an audit trail (5 minutes)
Create `tx["date"]` from `transaction_date`, using the explicit year-month-day format and `errors="coerce"`. Count missing source tokens separately from present tokens that failed parsing. Display the date range, assert zero missing parsed dates, and confirm that row count and raw date strings are unchanged.

Methods: `pd.to_datetime`, `.isna()`, `.notna()`, `.min()`, `.max()`, `.equals()`. Hint: a parse failure means **source present AND parsed result missing**.

As a small transfer check, construct a separate Series containing `"2025-02-28"`, `"2025-02-30"`, and `None`. Apply the same audit and report the two different reasons for `NaT`. Do not insert these tokens into `tx`.

Tiny unrelated example:
```python
parsed = pd.to_datetime(pd.Series(["2031-07-09"]), format="%Y-%m-%d")
```
Why would immediately dropping every `NaT` obscure a source-quality issue?

In [28]:
tx["date"] = pd.to_datetime(raw["transaction_date"], errors="coerce")

print("dates missing in original column")
print(raw["transaction_date"].isna().sum())
print("dates missing in new column")
print(tx["date"].isna().sum())

assert tx["date"].isna().sum() == 0, "There are missing dates in the new column"



print("Min date:", tx["date"].min())
print("Max date:", tx["date"].max())

dates missing in original column
0
dates missing in new column
0
Min date: 2024-01-01 00:00:00
Max date: 2025-12-30 00:00:00


Parsing seems fine, no NA's were introduced.

## Task 3 — Make the observation window explicit (4 minutes)
Define `start = pd.Timestamp("2025-01-01")` and `end = pd.Timestamp("2025-12-31")`. Select `start <= date < end` into `analysis` with `.copy()`. Create integer `month` using `.dt.month` and inspect monthly transaction counts.

Acceptance criteria: **5,983 transactions**, all within the interval; monthly counts sum to this total. Assert the interval and the reconciliation.

Explain why a half-open interval gives adjacent windows no overlap, and why December here is a partial calendar month. Do not interpret a low monthly count without considering the number of observed days.

In [36]:
# Define observation window
start = pd.Timestamp("2025-01-01")
end = pd.Timestamp("2025-12-31")

analysis = tx.copy()

analysis = analysis[(analysis["date"] >= start) & (analysis["date"] <= end)]
analysis["month"] = analysis["date"].dt.month
analysis.head()



,transaction_id,customer_id,transaction_date,category,amount_eur,discount_pct,returned,date,month
0,T000001,C00001,2025-03-23,hardware,118.37,0.0,False,2025-03-23,3
2,T000003,C00001,2025-11-23,training,103.17,0.0,False,2025-11-23,11
4,T000005,C00001,2025-06-30,software,249.17,0.0,False,2025-06-30,6
6,T000007,C00001,2025-08-31,services,348.22,0.0,False,2025-08-31,8
8,T000009,C00001,2025-01-05,software,191.81,0.0,False,2025-01-05,1


In [42]:
# Count transactions per month
monthly_transactions = analysis.groupby("month").agg(
    transaction_count=("month", "size")
).reset_index()

print(monthly_transactions)

print(monthly_transactions["transaction_count"].sum())

    month  transaction_count
0       1                538
1       2                434
2       3                518
3       4                486
4       5                549
5       6                490
6       7                507
7       8                520
8       9                464
9      10                520
10     11                467
11     12                490
5983


## Task 4 — UTC instants versus Vienna clock time (7 minutes)
Use this **independent synthetic timestamp probe**, not the date-only transaction column. Construct a DataFrame named `clock_probe` with these two string columns:

| session_id | timestamp_utc |
|---|---|
| S1 | 2025-03-30T00:30:00Z |
| S1 | 2025-03-30T01:30:00Z |

Parse `timestamp_utc` with `pd.to_datetime(..., utc=True)`, then create `vienna_time` with `.dt.tz_convert("Europe/Vienna")`. Before execution, predict whether conversion changes the represented instants or the row count. Display both aware timestamp columns, including offsets. Compute the elapsed hours between the two rows by subtracting the aware timestamps.

Acceptance criteria: **2 rows**, equal elapsed duration in UTC and Vienna, and a **one-hour elapsed gap** despite a two-hour jump in local clock labels. Explain this difference in one sentence.

API reminder: for a timedelta **scalar**, use `.total_seconds()`; for a timedelta **Series**, use `.dt.total_seconds()`.

Hint: `utc=True` is appropriate because these strings explicitly describe UTC instants. `tz_localize` attaches a zone to naive clock readings; `tz_convert` converts already-aware timestamps. A naive local time such as `2025-03-30 02:30` does not exist in Vienna, so a real import needs an explicit policy. Do not remove timezone information before calculating elapsed durations.

In [48]:
clock_probe = pd.DataFrame({
    "session_id" : ["S1", "S1"],
    "timestamp_utc" : ["2025-03-30T00:30:00Z", "2025-03-30T01:30:00Z"]
})
clock_probe["timestamp_utc"] = pd.to_datetime(clock_probe["timestamp_utc"], utc=True)
clock_probe["vienna_time"] = clock_probe["timestamp_utc"].dt.tz_convert("Europe/Vienna")
clock_probe['elapsed_time'] = clock_probe['vienna_time'] - clock_probe['vienna_time'].iloc[0]
print(clock_probe)

  session_id             timestamp_utc               vienna_time  \
0         S1 2025-03-30 00:30:00+00:00 2025-03-30 01:30:00+01:00   
1         S1 2025-03-30 01:30:00+00:00 2025-03-30 03:30:00+02:00   

     elapsed_time  
0 0 days 00:00:00  
1 0 days 01:00:00  


## Task 5 — Gaps within each customer (6 minutes)
Sort `analysis` by `customer_id`, `date`, then `transaction_id` into `ordered`. The final key only breaks same-day ties reproducibly; it does not reveal the true intraday order. Create `gap_days` using within-customer `.diff()` on the date column, followed by `.dt.days`.

Input grain and output grain are both **one transaction per row**. Desired output: the same 5,983 rows, with one missing gap at the first observed purchase per customer. Assert that every observed gap is nonnegative and that the missing-gap count equals the number of distinct customers in `analysis`.

First-exposure example with unrelated data:
```python
visits = pd.DataFrame({
    "person": ["A", "A", "B"],
    "day": pd.to_datetime(["2030-01-02", "2030-01-05", "2030-01-03"]),
})
visits["elapsed"] = visits.groupby("person")["day"].diff()
```

Inspect one customer with multiple transactions. Explain why a zero-day gap is valid and why the first missing gap does not mean zero days. Gaps here refer only to purchases observed in the selected window.

In [60]:
ordered = analysis.sort_values(by=["customer_id", "date", "transaction_id"])
ordered["gap_days"] = ordered.groupby("customer_id")["date"].diff().dt.days
ordered

,transaction_id,customer_id,transaction_date,category,amount_eur,discount_pct,returned,date,month,gap_days
8,T000009,C00001,2025-01-05,software,191.81,0.0,False,2025-01-05,1,NaN
0,T000001,C00001,2025-03-23,hardware,118.37,0.0,False,2025-03-23,3,77.0
4,T000005,C00001,2025-06-30,software,249.17,0.0,False,2025-06-30,6,99.0
6,T000007,C00001,2025-08-31,services,348.22,0.0,False,2025-08-31,8,62.0
2,T000003,C00001,2025-11-23,training,103.17,0.0,False,2025-11-23,11,84.0
...,...,...,...,...,...,...,...,...,...,...
12083,T012084,C01999,2025-10-16,services,97.93,0.0,False,2025-10-16,10,14.0
12089,T012090,C02000,2025-02-17,hardware,72.65,0.0,False,2025-02-17,2,NaN
12085,T012086,C02000,2025-04-30,services,873.22,0.0,False,2025-04-30,4,72.0
12090,T012091,C02000,2025-08-10,software,76.66,NaN,False,2025-08-10,8,102.0


## Task 6 — Recency at customer grain (6 minutes)
Build `customer_summary` from `analysis` with one row per customer observed in this window. Include `customer_id`, `n_transactions`, and `last_purchase` using named aggregation. Set `reference_date = pd.Timestamp("2025-12-31")` and calculate `recency_days` as the whole-day difference between that reference and `last_purchase`.

Desired output: **1,915 rows × 4 columns**, unique customer IDs, strictly positive recency, and transaction counts summing to 5,983. Assert these conditions and report median recency in days.

Methods: `.groupby(..., as_index=False).agg(...)`, timestamp subtraction, `.dt.days`. Hint: subtraction broadcasts a scalar timestamp over the Series.

Explain why this table excludes some customers from the full dataset and why a customer's first purchase in this window need not be their first-ever purchase. Would using today's date make the result reproducible?

In [73]:
reference_date = pd.Timestamp("2025-12-31")
customer_summary = analysis.groupby("customer_id").agg(
    n_transaction = ("transaction_id", "count"),
    last_purchase = ("date", "max")
).reset_index()
customer_summary["recency_days"] = reference_date - customer_summary["last_purchase"]
customer_summary["recency_days"] = customer_summary["recency_days"].dt.days
print("median recency in days", customer_summary["recency_days"].median())




# testing
assert customer_summary.shape[1] == 4, "customer_summary should have 4 columns"
assert customer_summary.shape[0] == 1915, "customer_summary should have 1915 rows"
#ok i get it


median recency in days 81.0


## Task 7 — From event rows to a daily calendar (6 minutes)
Count transactions by date into a Series. Build `calendar` with `pd.date_range(start=start, end=end, freq="D", inclusive="left")`. Reindex your count Series to this calendar with `fill_value=0`, name it `n_transactions`, and call it `daily_counts`.

Desired output: a sorted Series with a unique `DatetimeIndex`, **364 entries**, no missing values, and a sum of 5,983. Assert these invariants. Report the number of zero-transaction days you actually observe; do not assume there must be any.

Methods: `.groupby("date").size()`, `.reindex(calendar, fill_value=0)`, `.rename(...)`. Hint: `.size()` counts rows irrespective of missing values in other columns. The grain has changed from transactions to calendar days.

Why would filling absent dates with zero be wrong during an unrecorded system outage? Do not add December 31: it falls outside this source's observation interval.

In [74]:
analysis.groupby("date").agg(
    n_transaction = ("transaction_id", "count")
).reset_index()

,date,n_transaction
0,2025-01-01,23
1,2025-01-02,22
2,2025-01-03,13
3,2025-01-04,16
4,2025-01-05,14
...,...,...
359,2025-12-26,24
360,2025-12-27,14
361,2025-12-28,15
362,2025-12-29,12


## Task 8 — Audit a rolling-window suggestion (6 minutes)
An AI proposes this summary of your completed daily Series:
```python
smoothed = daily_counts.rolling(window=7, center=True, min_periods=1).mean()
```
Read it before running anything. Explain which future days can contribute to the value labeled January 4, and why the beginning of the series averages fewer than seven observations.

Create `daily` as a DataFrame from `daily_counts` and add `mean_7d` using a **trailing seven-day mean including the current day**, requiring all seven days (`center=False`, `min_periods=7`). Keep the date index for now.

Acceptance criteria: 364 rows; exactly the first six rolling values are missing; the seventh equals the mean of the first seven daily counts. Verify that last equality with a small absolute tolerance such as `1e-10`.

Hint: an integer window counts **rows**, so seven rows only means seven calendar days after establishing a complete daily calendar. Even this trailing result includes today's final count; it is not available at the start of that day.

## Task 9 — Plot and state a defensible finding (6 minutes)
Reset the date index into a column named `date` in a separate `plot_data` table. With plotnine, plot daily counts as a light line and the trailing mean as a stronger contrasting line. Label the axes with date and transactions per day. Leave the first six rolling means missing; use `na_rm=True` on that layer if needed.

Useful imports: `ggplot`, `aes`, `geom_line`, `labs`, `theme_minimal` from `plotnine`.

Small unrelated layer example:
```python
# Markdown example only: temperatures is a separate example table.
(ggplot(temperatures, aes(x="day", y="degrees"))
 + geom_line(color="steelblue")
 + labs(x="Day", y="Temperature (°C)")
 + theme_minimal())
```
For your two layers, provide a different `aes(y=...)` in each `geom_line`. Hint: `.rename_axis("date").reset_index()` preserves the index as a named column.

Report one numerical finding from your daily summary and interpret the plotted variability in two sentences. Explain why smoothing alone cannot establish seasonality, a marketing effect, or a forecast. This is synthetic data, so a flat-looking pattern is a valid finding.

## Retrieval practice — Task 10: select, sort, validate (5 minutes)
Using `customer_summary`, return the five customers with the largest `recency_days`. Break ties by ascending `customer_id`. Display only `customer_id`, `last_purchase`, and `recency_days`, in that order.

Before executing, predict the result's type and shape. Assert five unique IDs and descending recency. Leave the source summary unchanged. Explain what "most recent" versus "largest recency" means; which end of the sorted table represents each?

## Professional reasoning questions (3 minutes)
Answer briefly in a new markdown cell or print a multiline response below.
1. Explain why a global date difference after sorting by customer can create misleading gaps at customer boundaries. Which assertion or inspection would expose this?
2. What could go wrong if you treated the rows in `analysis` as independent customer observations in a statistical comparison?
3. Would the plotted seven-day mean be a valid feature for predicting today's count at midnight? Explain what information is available then and how you would shift the window.
4. What conclusion about customer churn is **not** justified by the recency table, and what additional observation or definition would you need?

## Self-check and reflection (2 minutes)
Record:
- Confidence from 1–5
- Hardest point
- One thing you can now explain
- One operation that deserves repetition
- Whether the hour felt too light, about right, or too heavy

Finish line: your cells run from a fresh kernel, invariants hold, and you can state the observation window and the row grain at each stage. Ask for review when ready.

## Optional extension — Observation gaps versus zero activity
**Outside the estimated core duration (15–20 minutes).** In a separate copy of the daily table, simulate an unavailable reporting interval from March 10 through March 12, 2025. Treat those dates as unknown counts, not zero, and recompute the trailing mean requiring seven observed days.

Compare with a second, deliberately incorrect version that fills the outage with zeros. Quantify how many rolling values each policy changes relative to the original, and plot the affected period. State why a completeness flag would help a downstream analyst.

Acceptance criteria: the full 364-day calendar remains; the first version has three additional missing daily counts and propagates uncertainty to all windows touching the outage. The original `daily` remains unchanged. No new data or download is needed.